# IoT Orange Experiments

Notebook orientado al caso practico del TFG: usar un dataset propio de mandarinas para preparar datos, documentar el flujo de entrenamiento con `anomalib` y dejar una plantilla clara para inferencia.


## Objetivo del cuaderno

- resumir el dataset propio usado en el caso IoT;
- dejar listas las utilidades de redimensionado y aumento de datos;
- documentar el flujo de entrenamiento sobre `Folder` y la alternativa con `MVTec`;
- dejar una plantilla segura para inferencia con `PaDiM`.


## Como leer este notebook

La idea no es forzar un entrenamiento largo cada vez que se abra el cuaderno, sino convertirlo en una pieza de apoyo legible. Por eso las celdas pesadas quedan encapsuladas como funciones o plantillas de ejecucion manual, mientras que las celdas ejecutadas se centran en resumir el estado del entorno y del dataset.


## Entorno y rutas


In [ ]:
from pathlib import Path
import importlib.util
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

DATA_DIR = ROOT / "data"
DOCS_DIR = ROOT / "docs"
ANOMALIB_DIR = ROOT / "anomalib"
RESULTS_DIR = ROOT / "results"
NOTES_DIR = ROOT / "notes"
NOTEBOOKS_DIR = ROOT / "notebooks"
NOTES_DIR.mkdir(exist_ok=True)

RAW_DATASET_DIR = DATA_DIR / "mandarins_pynq_raw"
CROPPED_DATASET_DIR = DATA_DIR / "mandarins_pynq_cropped"
AUGMENTED_DATASET_DIR = DATA_DIR / "mandarins_pynq_augmented"
INFERENCE_NORMAL_IMAGE = DATA_DIR / "inference_normal.png"
INFERENCE_ANOMALY_IMAGE = DATA_DIR / "inference_anomaly.png"

try:
    from anomalib.config import get_configurable_parameters as _anomalib_probe
    ANOMALIB_AVAILABLE = True
except Exception:
    ANOMALIB_AVAILABLE = False

TIMM_AVAILABLE = importlib.util.find_spec("timm") is not None
CV2_AVAILABLE = importlib.util.find_spec("cv2") is not None
KERAS_AVAILABLE = importlib.util.find_spec("keras") is not None

environment_summary = pd.DataFrame(
    [
        {"item": "repo_root", "value": str(ROOT)},
        {"item": "anomalib_available", "value": ANOMALIB_AVAILABLE},
        {"item": "timm_available", "value": TIMM_AVAILABLE},
        {"item": "cv2_available", "value": CV2_AVAILABLE},
        {"item": "keras_available", "value": KERAS_AVAILABLE},
        {"item": "raw_dataset_exists", "value": RAW_DATASET_DIR.exists()},
        {"item": "augmented_dataset_exists", "value": AUGMENTED_DATASET_DIR.exists()},
        {"item": "results_exists", "value": RESULTS_DIR.exists()},
    ]
)
environment_summary


Esta primera celda comprueba que las rutas canonicas del repo existen y, muy importante, inserta el directorio raiz del proyecto en `sys.path`. Con eso evitamos el falso negativo que aparecia al ejecutar el notebook desde `notebooks/` y que hacia pensar que `anomalib` no estaba disponible cuando en realidad si lo estaba en el entorno de trabajo.


## Resumen del dataset


In [ ]:
def dataset_count_table(dataset_dir: Path) -> pd.DataFrame:
    rows = []
    for label in ["normal", "abnormal"]:
        label_dir = dataset_dir / label
        count = len([path for path in sorted(label_dir.iterdir()) if path.is_file()]) if label_dir.exists() else 0
        rows.append({"dataset": dataset_dir.name, "label": label, "images": count})
    return pd.DataFrame(rows)


def summarize_mandarin_datasets() -> pd.DataFrame:
    frames = []
    for dataset_dir in [RAW_DATASET_DIR, CROPPED_DATASET_DIR, AUGMENTED_DATASET_DIR]:
        if dataset_dir.exists():
            frames.append(dataset_count_table(dataset_dir))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["dataset", "label", "images"])


def show_image_grid(image_paths, titles, figsize=(12, 4)) -> None:
    fig, axes = plt.subplots(1, len(image_paths), figsize=figsize)
    if len(image_paths) == 1:
        axes = [axes]
    for axis, image_path, title in zip(axes, image_paths, titles):
        axis.imshow(Image.open(image_path))
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()

summary_df = summarize_mandarin_datasets()
summary_df


Antes de entrenar nada conviene mirar si el dataset consolidado se parece a lo que describe la memoria del TFG: un conjunto pequeno, muy desbalanceado y con distintas versiones derivadas para pruebas de preprocesado y aumento de datos.


In [ ]:
example_paths = [
    next(iter(sorted((RAW_DATASET_DIR / "normal").iterdir()))),
    next(iter(sorted((RAW_DATASET_DIR / "abnormal").iterdir()))),
    INFERENCE_NORMAL_IMAGE,
    INFERENCE_ANOMALY_IMAGE,
]
show_image_grid(example_paths, ["Raw normal", "Raw abnormal", "Inference normal", "Inference anomaly"], figsize=(14, 4))


## Preparacion y aumento de datos


En el trabajo original esta parte servia para transformar un conjunto muy pequeno de imagenes en una base mas util para experimentar. Aqui mantenemos esa idea pero con una implementacion mas robusta: las funciones se cargan solo si las dependencias visuales estan disponibles y trabajan siempre con rutas relativas al repo.


In [ ]:
if not CV2_AVAILABLE or not KERAS_AVAILABLE:
    print("Se omiten las utilidades de preprocesado porque faltan cv2 o keras en el entorno.")
else:
    import cv2
    from keras.preprocessing.image import ImageDataGenerator
    from keras.utils import img_to_array, load_img
    from numpy import expand_dims

    def resize_images(source_dir: Path, target_dir: Path, image_size=(768, 768)) -> None:
        target_dir.mkdir(parents=True, exist_ok=True)
        for image_path in sorted(source_dir.iterdir()):
            if not image_path.is_file():
                continue
            image = np.array(Image.open(image_path))
            resized = cv2.resize(image, image_size)
            Image.fromarray(resized).save(target_dir / image_path.name)

    def resize_dataset(source_dataset: Path, target_dataset: Path, image_size=(768, 768)) -> None:
        resize_images(source_dataset / "normal", target_dataset / "normal", image_size=image_size)
        resize_images(source_dataset / "abnormal", target_dataset / "abnormal", image_size=image_size)

    def augment_image(image_path: Path, destination_dir: Path, augmentations: int) -> None:
        image = img_to_array(load_img(image_path))
        batch_source = expand_dims(image, 0)
        augmenter = ImageDataGenerator(
            rotation_range=40,
            brightness_range=[0.4, 1.3],
            horizontal_flip=True,
            vertical_flip=True,
            fill_mode="nearest",
        )
        generated = 0
        for _ in augmenter.flow(batch_source, batch_size=1, save_prefix="augmented", save_to_dir=str(destination_dir), save_format="jpg"):
            generated += 1
            if generated >= augmentations:
                break

    def build_augmented_dataset(source_dataset: Path = RAW_DATASET_DIR, target_dataset: Path = AUGMENTED_DATASET_DIR, augmentations_per_image: int = 10) -> None:
        target_dataset.mkdir(parents=True, exist_ok=True)
        for label in ["normal", "abnormal"]:
            destination = target_dataset / label
            destination.mkdir(parents=True, exist_ok=True)
            for image_path in sorted((source_dataset / label).iterdir()):
                if image_path.is_file():
                    Image.open(image_path).save(destination / image_path.name)
            base_images = [path for path in sorted(destination.iterdir()) if path.is_file()]
            for image_path in base_images:
                augment_image(image_path, destination, augmentations_per_image)

    print("Utilidades de preprocesado cargadas.")


## Plantilla de entrenamiento con anomalib


Este bloque no ejecuta un `fit` completo por defecto, pero deja preparada la logica que habria que lanzar para repetir el flujo del TFG. La separacion en funciones ayuda a que el cuaderno se pueda leer como documentacion del pipeline, no solo como historial de ejecucion.


In [ ]:
if not ANOMALIB_AVAILABLE:
    print("anomalib no esta disponible en este entorno.")
else:
    from pytorch_lightning import Trainer
    from anomalib.config import get_configurable_parameters
    from anomalib.data.folder import Folder
    from anomalib.data.mvtec import MVTec
    from anomalib.models import get_model
    
    def build_datamodule(use_mandarin_dataset: bool = True):
        if use_mandarin_dataset:
            datamodule = Folder(root=str(AUGMENTED_DATASET_DIR), image_size=768, seed=42)
        else:
            datamodule = MVTec(
                root=str(DATA_DIR / "mvtec_anomaly_detection"),
                category="bottle",
                image_size=256,
                train_batch_size=32,
                test_batch_size=32,
                num_workers=8,
                task="segmentation",
                seed=42,
            )
        datamodule.setup()
        return datamodule

    def build_experiment(model_name: str, use_mandarin_dataset: bool = True):
        config_path = ANOMALIB_DIR / "models" / model_name / "config.yaml"
        config = get_configurable_parameters(config_path=str(config_path))
        datamodule = build_datamodule(use_mandarin_dataset=use_mandarin_dataset)
        model = get_model(config)
        callbacks = []
        return datamodule, model, callbacks, config

    def metrics_to_frame(metrics) -> pd.DataFrame:
        return pd.DataFrame({"Metric": list(metrics.keys()), "Value": [float(value) for value in metrics.values()]})

    print("Plantilla de entrenamiento cargada. Se omiten callbacks avanzados para evitar dependencias opcionales como wandb en la exportacion.")


In [ ]:
training_notes = pd.DataFrame(
    [
        {"step": 1, "action": "Elegir modelo anomalib", "example": "patchcore / padim / fastflow"},
        {"step": 2, "action": "Construir datamodule", "example": "build_datamodule(use_mandarin_dataset=True)"},
        {"step": 3, "action": "Lanzar entrenamiento", "example": "Trainer.fit(...)"},
        {"step": 4, "action": "Evaluar metricas", "example": "trainer.test(...)"},
    ]
)
training_notes


## Plantilla de inferencia con PaDiM


La inferencia se deja separada del entrenamiento porque en el proyecto original dependia de checkpoints concretos. Al mantenerla como plantilla, el notebook sigue siendo util aunque el repositorio no incluya todos los pesos historicos.


In [ ]:
if not ANOMALIB_AVAILABLE:
    print("Se omite la plantilla de inferencia porque anomalib no esta disponible.")
else:
    from torch.utils.data import DataLoader
    from pytorch_lightning import Trainer
    from anomalib.data.inference import InferenceDataset
    from anomalib.models.padim.lightning_model import PadimLightning
    from anomalib.pre_processing.transforms import Denormalize

    def run_padim_inference(image_path: Path, checkpoint_path: Path):
        model = PadimLightning.load_from_checkpoint(str(checkpoint_path))
        trainer = Trainer()
        dataloader = DataLoader(InferenceDataset(str(image_path), image_size=(256, 256)))
        return trainer.predict(model=model, dataloaders=dataloader)[0]

    def plot_prediction(output) -> None:
        image = Denormalize()(output["image"][0])
        anomaly_map = output["anomaly_maps"][0].cpu().numpy().squeeze()
        pred_mask = output["pred_masks"][0].cpu().numpy().squeeze()
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(image)
        axes[0].set_title("Image")
        axes[1].imshow(anomaly_map)
        axes[1].set_title("Anomaly map")
        axes[2].imshow(pred_mask)
        axes[2].set_title("Predicted mask")
        for axis in axes:
            axis.axis("off")
        plt.tight_layout()
        plt.show()

    print("Funciones de inferencia cargadas. Ejecuta manualmente con un checkpoint valido en results/.")


## Cierre

Este cuaderno queda como pieza de apoyo para la parte IoT del TFG: resume el dataset propio, explica el papel del aumento de datos y deja listo el flujo para repetir entrenamiento e inferencia de forma ordenada.
